# A3 · nb06 — report figures and tables

**CS4.406 Assignment 3**  ·  prerequisites: `nb00`–`nb05` complete

Pure aggregation — no benchmarking. Everything here is derived from the single
`results.csv` that nb01–nb05 appended to, so every number in the report traces back to one
measured row. Runtime ≈ 2 minutes.

Produces:

| output | feeds |
|---|---|
| `fig1_sift_tradeoff.png`, `fig2_glove_tradeoff.png` | Part 2: recall vs QPS per dataset |
| `table1_qps_at_recall.md/.csv` | Part 2: QPS at recall 0.90 and 0.95 |
| `table2_build_memory.md/.csv` | Part 2: build time, size, bytes/vector at 0.95 |
| `fig3_part3_scaling.png` | Part 3 |
| `fig4_part4_geometry.png` | Part 4 (re-rendered from the table for consistency) |
| `report_numbers.json` | every figure quoted in the report's prose |

### Two data-hygiene steps, stated up front

1. **Duplicate rows are removed.** The first version of nb05 logged each d=1024 fixed-budget
   point twice — once as the fixed measurement, once as the first point of the H4.4 sweep.
   The values are identical, but left in place they would double-count. Dedup keys on
   (part, dataset, n_vectors, index_kind, build_params, search_param, value), keeping the last.
2. **Part 2 and Part 3 numbers are never mixed in one table.** They answer different
   questions: Part 2 interpolates QPS *to* recall exactly 0.95 along a swept curve, while
   Part 3 reports the cheapest *grid* setting whose tune-split recall reached 0.95, which
   generally overshoots (e.g. HNSW at 1M: 5817 qps interpolated vs 4237 qps measured at
   recall 0.9712). Both are correct; conflating them is not.

## 1 · Setup

In [ ]:
import subprocess, sys

def pip(*a):
    print(subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a],
                         capture_output=True, text=True).stdout[-400:] or "ok")
for m, p in [("huggingface_hub", "huggingface_hub")]:
    try: __import__(m)
    except ImportError: pip(p)

In [ ]:
%%writefile annbench_core.py
"""
annbench_core.py -- shared core for CS4.406 Assignment 3 (ANN benchmarking).

Every notebook writes an identical copy of this file, so each notebook runs standalone.

Conventions used everywhere in this project:
  * All distances are SQUARED L2, computed on float32 vectors. GloVe is unit-normalised
    first, so squared-L2 order == cosine order. We never rely on the scale of the
    distances stored in the ann-benchmarks HDF5 files (euclidean vs cosine units);
    those files are used for their neighbour IDS only, as an independent cross-check.
  * k = 10 everywhere.
  * Queries 0..4999  = TUNE split (parameter picking).
    Queries 5000..9999 = EVAL split (every number that appears in the report).
  * Timing is single-thread, one query at a time. See bench_single_query.
"""

import json
import os
import platform
import re
import subprocess
import sys
import time
import hashlib

import numpy as np

K = 10
TIE_EPS = 1e-6
TUNE = slice(0, 5000)
EVAL = slice(5000, 10000)


def set_split(n_queries):
    """Re-point the tune/eval halves. The full datasets ship 10,000 queries, but quick mode
    uses fewer, and the halves must move with them so no reported number is ever measured
    on a query that was used for tuning."""
    global TUNE, EVAL
    half = n_queries // 2
    TUNE, EVAL = slice(0, half), slice(half, n_queries)
    return TUNE, EVAL

HF_REPO = "donbosoc/ire-a3-ann"
HF_REPO_TYPE = "dataset"

def _pick_scratch():
    """Scratch for big files: /kaggle/temp does not count against the 20 GB output quota,
    but it is not guaranteed to exist or be writable, so fall back rather than crash.
    A3_SCRATCH overrides it entirely, which is how run_all.sh keeps everything in-tree."""
    for cand in ([os.environ["A3_SCRATCH"]] if os.environ.get("A3_SCRATCH") else []) + [
            "/kaggle/temp/a3", "/kaggle/working/tmp/a3", "/tmp/a3"]:
        try:
            os.makedirs(cand, exist_ok=True)
            probe = os.path.join(cand, ".w")
            with open(probe, "w") as f:
                f.write("x")
            os.remove(probe)
            return cand
        except Exception:
            continue
    raise RuntimeError("no writable scratch directory found")


TMP = _pick_scratch()
OUT = os.environ.get("A3_OUT") or ("/kaggle/working" if os.path.isdir("/kaggle/working")
                                   else os.path.abspath("./out"))


def ensure_dirs():
    for p in (TMP, f"{TMP}/data", f"{TMP}/gt", f"{TMP}/idx",
              f"{OUT}/results", f"{OUT}/figures", f"{OUT}/logs", f"{OUT}/env"):
        os.makedirs(p, exist_ok=True)


# --------------------------------------------------------------------------------------
# Environment
# --------------------------------------------------------------------------------------

def env_fingerprint(verbose=True):
    """Capture the machine + library versions behind every measurement.

    Kaggle VMs are shared and the CPU model can differ between sessions, so this goes
    into every results row and the report states it rather than claiming a quiet machine.
    """
    info = {}
    try:
        cpuinfo = open("/proc/cpuinfo").read()
        models = re.findall(r"^model name\s*:\s*(.+)$", cpuinfo, re.M)
        info["cpu_model"] = models[0].strip() if models else "unknown"
        info["cpu_logical"] = len(models)
    except Exception:
        info["cpu_model"], info["cpu_logical"] = platform.processor() or "unknown", os.cpu_count()
    try:
        mem_kb = int(re.search(r"MemTotal:\s*(\d+)", open("/proc/meminfo").read()).group(1))
        info["mem_gb"] = round(mem_kb / 1048576, 1)
    except Exception:
        info["mem_gb"] = None

    info["python"] = sys.version.split()[0]
    info["numpy"] = np.__version__
    for mod in ("faiss", "torch", "h5py"):
        try:
            info[mod] = __import__(mod).__version__
        except Exception:
            info[mod] = None
    try:
        import torch
        info["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    except Exception:
        info["gpu"] = None
    try:
        info["nvidia_smi"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception:
        info["nvidia_smi"] = None

    # Stable id for "which machine produced this row".
    key = f"{info['cpu_model']}|{info['cpu_logical']}|{info['mem_gb']}"
    info["machine_id"] = hashlib.sha1(key.encode()).hexdigest()[:8]

    if verbose:
        print(f"machine_id : {info['machine_id']}")
        print(f"cpu        : {info['cpu_model']}  ({info['cpu_logical']} logical)")
        print(f"memory     : {info['mem_gb']} GB")
        print(f"gpu        : {info['gpu']}")
        print(f"versions   : python {info['python']}, numpy {info['numpy']}, "
              f"faiss {info['faiss']}, torch {info['torch']}, h5py {info['h5py']}")
    return info


def set_single_thread():
    """Pin FAISS to one thread. Every benchmark number in this project is single-thread."""
    import faiss
    faiss.omp_set_num_threads(1)
    n = faiss.omp_get_max_threads()
    assert n == 1, f"expected 1 FAISS thread, got {n}"
    return n


# --------------------------------------------------------------------------------------
# Hugging Face
# --------------------------------------------------------------------------------------

_TOKEN_NAMES = ["HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACEHUB_API_TOKEN",
                "HF", "hf_token", "huggingface"]


def hf_enabled():
    """HF mirroring is a convenience for running across Kaggle sessions, never a
    requirement. Everything this project writes also lands on local disk, so with no token
    (the normal case for someone reproducing the results) the mirror is simply skipped."""
    if os.environ.get("A3_NO_HF"):
        return False
    try:
        get_hf_token()
        return True
    except Exception:
        return False


def get_hf_token():
    """Find the HF token in Kaggle secrets, then the environment. Raises if absent."""
    tried = []
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in _TOKEN_NAMES:
            try:
                tok = client.get_secret(name)
                if tok:
                    print(f"HF token: Kaggle secret '{name}'")
                    return tok.strip()
            except Exception:
                tried.append(f"secret:{name}")
    except Exception:
        tried.append("kaggle_secrets unavailable")
    for name in _TOKEN_NAMES:
        tok = os.environ.get(name)
        if tok:
            print(f"HF token: environment variable '{name}'")
            return tok.strip()
        tried.append(f"env:{name}")
    raise RuntimeError(
        "No Hugging Face token found. Add it under Notebook > Add-ons > Secrets with "
        f"label 'HF_TOKEN' and attach it to this notebook. Tried: {tried}")


def hf_api(token=None):
    from huggingface_hub import HfApi
    return HfApi(token=token or get_hf_token())


def hf_ensure_repo(token=None):
    api = hf_api(token)
    api.create_repo(HF_REPO, repo_type=HF_REPO_TYPE, exist_ok=True, private=False)
    return api


def hf_upload(local_path, path_in_repo, token=None, msg=None, quiet=False, timeout=420):
    """Upload one file, but never let a network stall hang the session.

    huggingface_hub has no hard wall-clock cap on an upload, and a stalled connection will
    block forever. A 12-hour Kaggle session was lost to exactly that. The work runs on a
    daemon thread with a join timeout: if it overruns we warn and carry on, because the
    local copy of everything we upload is always intact on disk.
    """
    import threading

    if not hf_enabled():
        return False
    size = os.path.getsize(local_path) / 1e6
    if not quiet:
        print(f"  -> hf://{HF_REPO}/{path_in_repo}  ({size:.1f} MB)", flush=True)

    box = {}

    def work():
        try:
            hf_api(token).upload_file(
                path_or_fileobj=local_path, path_in_repo=path_in_repo,
                repo_id=HF_REPO, repo_type=HF_REPO_TYPE,
                commit_message=msg or f"add {path_in_repo}")
            box["ok"] = True
        except Exception as exc:
            box["err"] = exc

    th = threading.Thread(target=work, daemon=True)
    th.start()
    # Big files legitimately take longer, so scale the allowance with size.
    th.join(timeout + size * 2)
    if th.is_alive():
        print(f"  ! upload of {path_in_repo} still running after "
              f"{timeout + size * 2:.0f}s -- continuing; the local copy is intact",
              flush=True)
        return False
    if "err" in box:
        print(f"  ! upload of {path_in_repo} failed: {box['err']}", flush=True)
        return False
    return True


def hf_download(path_in_repo, token=None, local_dir=None):
    """Return a local path for a repo file, or None if it is not in the repo."""
    from huggingface_hub import hf_hub_download
    if not hf_enabled():
        return None
    try:
        return hf_hub_download(repo_id=HF_REPO, repo_type=HF_REPO_TYPE,
                               filename=path_in_repo, token=token or get_hf_token(),
                               local_dir=local_dir or f"{TMP}/hf")
    except Exception as e:
        if "404" in str(e) or "EntryNotFound" in type(e).__name__:
            return None
        raise


def fetch(repo_path, token=None, verbose=True):
    """Download one prepared artifact from the HF mirror, erroring clearly if nb00 never ran."""
    p = hf_download(repo_path, token=token, local_dir=f"{TMP}/hf")
    if p is None:
        raise FileNotFoundError(
            f"hf://{HF_REPO}/{repo_path} not found -- run nb00_setup_data first")
    if verbose:
        print(f"  {repo_path:44s} {human(os.path.getsize(p))}")
    return p


def load_local(name, with_base=True, with_stored=False):
    """Load what prepare_data.py wrote to local disk (the path run_all.sh uses)."""
    d = {}
    if with_base:
        d["base"] = np.load(f"{TMP}/data/{name}_base.npy")
    d["query"] = np.load(f"{TMP}/data/{name}_query.npy")
    d["gt_ids"] = np.load(f"{TMP}/gt/{name}_top100_ids.npy")
    d["gt_dists"] = np.load(f"{TMP}/gt/{name}_top100_dists.npy")
    if with_stored:
        d["stored"] = np.load(f"{TMP}/data/{name}_stored_neighbors.npy")
    return d


def load_prepared(name, token=None, with_base=True, with_stored=False, verbose=True):
    """Load a dataset prepared by nb00: base, all 10k queries, and our exact top-100 GT.

    Use d["query"][TUNE] for parameter picking and d["query"][EVAL] for reported numbers.
    """
    d = {}
    if with_base:
        d["base"] = np.load(fetch(f"data/{name}_base.npy", token, verbose))
    d["query"] = np.load(fetch(f"data/{name}_query.npy", token, verbose))
    d["gt_ids"] = np.load(fetch(f"gt/{name}_top100_ids.npy", token, verbose))
    d["gt_dists"] = np.load(fetch(f"gt/{name}_top100_dists.npy", token, verbose))
    if with_stored:
        d["stored"] = np.load(fetch(f"data/{name}_stored_neighbors.npy", token, verbose))
    return d


# --------------------------------------------------------------------------------------
# Exact ground truth on the GPU
# --------------------------------------------------------------------------------------

def exact_gt_gpu(base, queries, k=100, chunk=None, mem_budget_gb=None,
                 refine=True, verbose=True):
    """Brute-force top-k by squared L2, selected on the GPU, then refined exactly.

    The GPU pass uses the (|q|^2 + |b|^2 - 2 q.b) expansion in float32, which is fast but
    loses a few ULPs. Selection errors from that can only happen between points that are
    equal to within those ULPs -- i.e. ties, which our recall metric forgives anyway. The
    returned DISTANCES are then recomputed by direct subtraction in float64, so the
    threshold that recall@10 depends on is exact.

    The distance block is (nq, chunk) float32, which at 10k queries is 400 MB per 10k
    base vectors, so a fixed chunk would OOM a 16 GB T4 outright. `chunk` is derived from
    the free VRAM and the query count unless given explicitly, and the block is written
    into ONE preallocated buffer with in-place ops -- writing the expression out as
    `q_sq + b_sq - 2*q@b.T` would allocate three such blocks per iteration instead.

    Returns (ids int32 (nq,k), dists float64 (nq,k)) sorted ascending.
    """
    import torch
    assert base.dtype == np.float32 and queries.dtype == np.float32
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    n, d = base.shape
    nq = queries.shape[0]
    k = min(k, n)

    # torch.topk allocates a workspace about the size of its input, so the real peak is
    # ~2x the distance block. Budget a fifth of free VRAM to leave headroom for that.
    if mem_budget_gb is None:
        if dev == "cuda":
            torch.cuda.empty_cache()
            mem_budget_gb = max(0.25, 0.20 * torch.cuda.mem_get_info()[0] / (1 << 30))
        else:
            mem_budget_gb = 2.0
    auto = max(1024, int(mem_budget_gb * (1 << 30) / (nq * 4)))
    chunk = min(chunk, auto) if chunk else auto
    chunk = min(chunk, n)

    q = torch.from_numpy(np.ascontiguousarray(queries).copy()).to(dev)
    q_sq = (q * q).sum(1, keepdim=True)
    buf = torch.empty((nq, chunk), device=dev, dtype=torch.float32)

    best_d = best_i = None
    t0 = time.perf_counter()
    if verbose:
        print(f"    gt: {n:,} x {nq:,} in blocks of {chunk:,} "
              f"(~{nq * chunk * 4 / 1e9:.2f} GB/block) on {dev}")
    for s in range(0, n, chunk):
        blk = torch.from_numpy(np.ascontiguousarray(base[s:s + chunk]).copy()).to(dev)
        m = blk.shape[0]
        dist = buf[:, :m]
        torch.mm(q, blk.T, out=dist)
        dist.mul_(-2.0).add_(q_sq).add_((blk * blk).sum(1)[None, :]).clamp_(min=0)
        kk = min(k, m)
        dv, di = torch.topk(dist, kk, dim=1, largest=False)
        di = di + s
        if best_d is None:
            best_d, best_i = dv.clone(), di
        else:
            cd, ci = torch.cat([best_d, dv], 1), torch.cat([best_i, di], 1)
            best_d, pos = torch.topk(cd, k, dim=1, largest=False)
            best_i = torch.gather(ci, 1, pos)
            del cd, ci, pos
        del blk, dv, di
        if verbose:
            print(f"\r    gt: {min(s + chunk, n):,}/{n:,} vectors", end="", flush=True)
    if verbose:
        print(f"\r    gt: {n:,}/{n:,} vectors in {time.perf_counter() - t0:.1f}s on {dev}")

    ids = best_i.cpu().numpy().astype(np.int32)
    approx = best_d.cpu().numpy().astype(np.float64)
    del q, q_sq, best_i, best_d, buf
    if dev == "cuda":
        torch.cuda.empty_cache()
    if not refine:
        return ids, approx

    dists = exact_dists_for_ids(base, queries, ids)
    order = np.argsort(dists, axis=1, kind="stable")
    ids = np.take_along_axis(ids, order, axis=1)
    dists = np.take_along_axis(dists, order, axis=1)
    return ids, dists


def exact_gt(base, queries, k=100, verbose=True):
    """Exact top-k, on the GPU when there is one and through FAISS when there is not.

    The GPU path is far faster for the big ground-truth jobs, but torch is not a hard
    dependency: FAISS's own flat index gives the same ids, and in both cases the returned
    distances are recomputed exactly in float64 so the recall threshold is not approximate.
    """
    try:
        import torch
        if torch.cuda.is_available():
            return exact_gt_gpu(base, queries, k=k, verbose=verbose)
    except ImportError:
        pass

    import faiss
    if verbose:
        print(f"    gt: {len(base):,} x {len(queries):,} via faiss (no GPU)", flush=True)
    ix = faiss.IndexFlatL2(base.shape[1])
    ix.add(base)
    _, ids = ix.search(np.ascontiguousarray(queries), min(k, len(base)))
    ids = ids.astype(np.int32)
    dists = exact_dists_for_ids(base, queries, ids)
    order = np.argsort(dists, axis=1, kind="stable")
    return np.take_along_axis(ids, order, axis=1), np.take_along_axis(dists, order, axis=1)


def exact_dists_for_ids(base, queries, ids, block=512):
    """Exact squared-L2 distance from each query to each of its listed ids, in float64.

    Direct subtraction (no q^2+b^2-2qb expansion), so there is no cancellation error.
    ids == -1 (index returned fewer than k) yields +inf. Chunked over queries to bound
    the (block, k, d) temporary.
    """
    nq, kk = ids.shape
    out = np.empty((nq, kk), dtype=np.float64)
    for s in range(0, nq, block):
        e = min(s + block, nq)
        idx = ids[s:e]
        valid = idx >= 0
        vecs = base[np.where(valid, idx, 0)].astype(np.float64)   # (b, k, d)
        diff = vecs - queries[s:e].astype(np.float64)[:, None, :]
        out[s:e] = np.where(valid, np.einsum("bkd,bkd->bk", diff, diff), np.inf)
    return out


# --------------------------------------------------------------------------------------
# Recall@10, tie-aware  (Part 1 asks for this specifically)
# --------------------------------------------------------------------------------------

def recall_at_k(retrieved_ids, retrieved_true_d, gt_dists, k=K, eps=TIE_EPS):
    """Per-query recall@k that does not punish ties at the k-th neighbour.

    A returned id counts as correct when its EXACT distance is <= the exact distance of
    the true k-th neighbour (plus a tiny slack). If the true 10th and 11th neighbours sit
    at the same distance, returning either one is right, so membership of the stored
    top-10 id list is the wrong test -- distance is the right one.

    Duplicate ids are credited once, so an index that returns the same vector twice
    cannot inflate its score.

    retrieved_ids    : (nq, k') ids from the index; -1 means "fewer than k' results".
    retrieved_true_d : (nq, k') EXACT squared-L2 distances for those ids (inf for -1).
    gt_dists         : (nq, >=k) exact squared-L2 distances of the true neighbours, ascending.
    returns          : (nq,) float64 in [0, 1].
    """
    retrieved_ids = np.asarray(retrieved_ids)
    retrieved_true_d = np.asarray(retrieved_true_d, dtype=np.float64)
    gt_dists = np.asarray(gt_dists, dtype=np.float64)
    if retrieved_ids.ndim == 1:
        retrieved_ids = retrieved_ids[None, :]
        retrieved_true_d = retrieved_true_d[None, :]
        gt_dists = gt_dists[None, :]
    assert retrieved_ids.shape == retrieved_true_d.shape
    assert gt_dists.shape[1] >= k, f"need >= {k} ground-truth distances"

    dk = gt_dists[:, k - 1][:, None]
    thresh = dk * (1.0 + eps) + eps           # relative slack, plus absolute for dk == 0
    hit = (retrieved_ids >= 0) & (retrieved_true_d <= thresh)

    # Count DISTINCT hit ids per row.
    masked = np.where(hit, retrieved_ids, -1)
    masked = np.sort(masked, axis=1)
    dup = np.zeros(masked.shape, dtype=bool)
    dup[:, 1:] = masked[:, 1:] == masked[:, :-1]
    n_hit = ((masked >= 0) & ~dup).sum(1)
    return np.minimum(n_hit, k) / float(k)


def recall_from_ids(retrieved_ids, base, queries, gt_dists, k=K):
    """recall_at_k convenience wrapper that recomputes the exact distances itself.

    Needed for IVF-PQ, whose returned distances are approximations of the real ones and
    must never be fed to the recall metric.
    """
    d = exact_dists_for_ids(base, queries, np.asarray(retrieved_ids))
    return recall_at_k(retrieved_ids, d, gt_dists, k=k)


# --------------------------------------------------------------------------------------
# Timing
# --------------------------------------------------------------------------------------

def bench_single_query(search_fn, queries, repeats=3, warmup=50):
    """Time `search_fn` one query at a time, single thread; report the median run.

    Per-query timers cost ~50-100 ns each against queries of 10^2-10^4 us, i.e. under
    0.1%, so QPS is taken as n / sum(per-query latencies) from the same pass that
    produces the latency percentiles.
    """
    nq = len(queries)
    for i in range(min(warmup, nq)):
        search_fn(queries[i:i + 1])

    runs = []
    for _ in range(repeats):
        lat = np.empty(nq, dtype=np.float64)
        for i in range(nq):
            t0 = time.perf_counter_ns()
            search_fn(queries[i:i + 1])
            lat[i] = time.perf_counter_ns() - t0
        runs.append(lat)

    qps = np.array([nq / (r.sum() / 1e9) for r in runs])
    med = runs[int(np.argsort(qps)[len(qps) // 2])]       # latencies of the median run
    return dict(
        qps_median=float(np.median(qps)),
        qps_runs=[round(float(x), 2) for x in qps],
        qps_spread_pct=round(float(100 * (qps.max() - qps.min()) / np.median(qps)), 2),
        lat_p50_ms=float(np.percentile(med, 50) / 1e6),
        lat_p95_ms=float(np.percentile(med, 95) / 1e6),
        n_queries=int(nq), repeats=int(repeats),
    )


def bench_batch(search_fn, queries, repeats=3, warmup=1):
    """Time one call that passes every query at once (Part 1's batched comparison)."""
    nq = len(queries)
    for _ in range(warmup):
        search_fn(queries)
    secs = []
    for _ in range(repeats):
        t0 = time.perf_counter_ns()
        search_fn(queries)
        secs.append((time.perf_counter_ns() - t0) / 1e9)
    secs = np.array(secs)
    return dict(qps_median=float(nq / np.median(secs)),
                qps_runs=[round(float(nq / s), 2) for s in secs],
                wall_s_median=float(np.median(secs)),
                n_queries=int(nq), repeats=int(repeats))


# --------------------------------------------------------------------------------------
# Results table
# --------------------------------------------------------------------------------------

COLUMNS = [
    "run_id", "timestamp", "part", "dataset", "n_vectors", "dim", "metric",
    "index_kind", "build_params", "search_param_name", "search_param_value",
    "recall_at_10", "recall_std", "qps_median", "qps_runs", "qps_spread_pct",
    "lat_p50_ms", "lat_p95_ms", "build_time_s", "train_time_s", "add_time_s",
    "index_size_bytes", "bytes_per_vector", "n_queries", "repeats", "threads",
    "query_split", "machine_id", "cpu_model", "faiss_version", "scanned_frac",
    "rc_at_10", "notes",
]


class ResultsLog:
    """Append-only results table, mirrored to HF after every few rows.

    One row per benchmark run, as the deliverable requires. Resumability comes from this
    file: `done()` lets a restarted notebook skip measurements already in the table,
    which is what makes a killed Kaggle session cheap.
    """

    KEY = ("part", "dataset", "n_vectors", "index_kind", "build_params",
           "search_param_name", "search_param_value")

    def __init__(self, env, token=None, path=f"{OUT}/results/results.csv",
                 repo_path="results/results.csv", flush_every=5, pull=True):
        import pandas as pd
        self.pd, self.env, self.token = pd, env, token
        self.path, self.repo_path, self.flush_every = path, repo_path, flush_every
        self._since_flush = 0
        os.makedirs(os.path.dirname(path), exist_ok=True)

        df = None
        if pull and hf_enabled():
            try:
                remote = hf_download(repo_path, token=token)
                if remote:
                    df = pd.read_csv(remote)
                    print(f"results.csv: pulled {len(df)} existing rows from HF")
            except Exception as e:
                print(f"results.csv: could not pull from HF ({e}); starting fresh")
        if df is None and os.path.exists(path):
            df = pd.read_csv(path)
            print(f"results.csv: {len(df)} existing rows found locally")
        if df is None:
            df = pd.DataFrame(columns=COLUMNS)
            print("results.csv: starting a new table")
        for c in COLUMNS:
            if c not in df.columns:
                df[c] = None
        self.df = df[COLUMNS]
        self.df.to_csv(self.path, index=False)

    @staticmethod
    def _sv(x):
        """Normalise a search-param value for comparison: pandas may read an all-numeric
        column back as float, turning 40 into '40.0' and breaking resume detection."""
        t = str(x)
        if t.endswith(".0") and t[:-2].lstrip("-").isdigit():
            t = t[:-2]
        return t

    @staticmethod
    def _norm(params):
        return json.dumps(params, sort_keys=True, separators=(",", ":")) \
            if isinstance(params, dict) else str(params)

    def done(self, part, dataset, n_vectors, index_kind, build_params,
             search_param_name, search_param_value):
        """True when this exact measurement is already in the table."""
        if len(self.df) == 0:
            return False
        want = (str(part), str(dataset), self._sv(n_vectors), str(index_kind),
                self._norm(build_params), str(search_param_name),
                self._sv(search_param_value))
        have = self.df[list(self.KEY)].astype(str).apply(
            lambda col: col.map(self._sv) if col.name in ("n_vectors", "search_param_value")
            else col)
        return bool((have == self.pd.Series(want, index=list(self.KEY))).all(axis=1).any())

    def add(self, *, part, dataset, n_vectors, dim, metric, index_kind, build_params,
            search_param_name, search_param_value, recall, timing, build_time_s=None,
            train_time_s=None, add_time_s=None, index_size_bytes=None,
            query_split="eval", threads=1, scanned_frac=None, rc_at_10=None,
            notes="", flush=False):
        recall = np.asarray(recall, dtype=np.float64)
        timing = timing or {}
        row = {
            "run_id": f"{int(time.time() * 1000)}-{np.random.randint(1e6):06d}",
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
            "part": part, "dataset": dataset, "n_vectors": int(n_vectors), "dim": int(dim),
            "metric": metric, "index_kind": index_kind,
            "build_params": self._norm(build_params),
            "search_param_name": search_param_name, "search_param_value": search_param_value,
            "recall_at_10": round(float(recall.mean()), 6),
            "recall_std": round(float(recall.std()), 6),
            "qps_median": (round(float(timing["qps_median"]), 3)
                           if timing.get("qps_median") is not None else None),
            "qps_runs": json.dumps(timing.get("qps_runs")),
            "qps_spread_pct": timing.get("qps_spread_pct"),
            "lat_p50_ms": timing.get("lat_p50_ms"),
            "lat_p95_ms": timing.get("lat_p95_ms"),
            "build_time_s": build_time_s, "train_time_s": train_time_s,
            "add_time_s": add_time_s, "index_size_bytes": index_size_bytes,
            "bytes_per_vector": (round(index_size_bytes / n_vectors, 2)
                                 if index_size_bytes else None),
            "n_queries": timing.get("n_queries"), "repeats": timing.get("repeats"),
            "threads": int(threads), "query_split": query_split,
            "machine_id": self.env.get("machine_id"), "cpu_model": self.env.get("cpu_model"),
            "faiss_version": self.env.get("faiss"), "scanned_frac": scanned_frac,
            "rc_at_10": rc_at_10, "notes": notes,
        }
        # .loc[] append rather than concat: concat against a frame holding all-NA columns
        # raises a FutureWarning on every single row, which floods the notebook log.
        self.df.loc[len(self.df)] = self.pd.Series(row)
        self.df.to_csv(self.path, index=False)
        self._since_flush += 1
        if flush or self._since_flush >= self.flush_every:
            self.flush()
        return row

    def flush(self):
        self._since_flush = 0
        if not hf_enabled():
            return
        t0 = time.time()
        try:
            hf_upload(self.path, self.repo_path, token=self.token,
                      msg=f"results.csv: {len(self.df)} rows", quiet=True)
        except Exception as e:
            print(f"  ! results.csv upload failed ({e}); the local copy is intact")
        dt = time.time() - t0
        if dt > 30:
            print(f"  (results.csv sync took {dt:.0f}s)", flush=True)


# --------------------------------------------------------------------------------------
# Misc
# --------------------------------------------------------------------------------------

def index_size_bytes(index, path=None):
    """Serialise a FAISS index to disk and measure it. Memory footprint ~= file size for
    these index types, and it is the one number we can take reproducibly."""
    import faiss
    path = path or f"{TMP}/idx/_size_probe.faiss"
    faiss.write_index(index, path)
    n = os.path.getsize(path)
    os.remove(path)
    return n


def normalise(x):
    """Unit-normalise rows in place-ish. Zero rows are left as zeros."""
    x = np.ascontiguousarray(x, dtype=np.float32)
    nrm = np.linalg.norm(x, axis=1, keepdims=True)
    nrm[nrm == 0] = 1.0
    return (x / nrm).astype(np.float32)


def relative_contrast(gt_dists, mean_dist_sq, k=1):
    """RC = (mean distance to a random point) / (distance to the k-th neighbour).

    Reported on true (non-squared) distances, the usual convention. Values near 1 mean
    the nearest neighbour is barely nearer than anything else, which is exactly when
    graph and cell indexes stop working.
    """
    dk = np.sqrt(np.asarray(gt_dists, dtype=np.float64)[:, k - 1])
    dk = np.where(dk == 0, np.finfo(np.float64).tiny, dk)
    return np.sqrt(np.asarray(mean_dist_sq, dtype=np.float64)) / dk


def mean_sq_dist_to_random(base, queries, n_sample=100_000, seed=0):
    """Mean squared distance from each query to a random sample of the base."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(base), size=min(n_sample, len(base)), replace=False)
    sample = base[np.sort(idx)].astype(np.float32)
    s_sq = (sample * sample).sum(1)
    out = np.empty(len(queries), dtype=np.float64)
    for s in range(0, len(queries), 512):
        e = min(s + 512, len(queries))
        q = queries[s:e].astype(np.float32)
        d = (q * q).sum(1)[:, None] + s_sq[None, :] - 2.0 * (q @ sample.T)
        out[s:e] = np.maximum(d, 0).mean(1)
    return out


def human(n):
    for unit in ("B", "KB", "MB", "GB"):
        if abs(n) < 1024 or unit == "GB":
            return f"{n:.1f} {unit}"
        n /= 1024

In [ ]:
import os, sys, json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter
sys.path.insert(0, os.getcwd())
import annbench_core as C

C.ensure_dirs()
ENV = C.env_fingerprint(verbose=False)
TOKEN = C.get_hf_token()

raw = pd.read_csv(C.fetch("results/results.csv", TOKEN))
print(f"pulled {len(raw)} rows")

KEY = ["part", "dataset", "n_vectors", "index_kind", "build_params",
       "search_param_name", "search_param_value"]
df = raw.drop_duplicates(subset=KEY, keep="last").copy()
print(f"deduplicated: {len(raw)} -> {len(df)} rows ({len(raw) - len(df)} duplicates dropped)")
df = df[df.search_param_value.astype(str) != "BUDGET_STOP"]
for c in ("recall_at_10", "qps_median", "build_time_s", "index_size_bytes", "scanned_frac"):
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
df["part"] = df["part"].astype(str)
print(f"usable: {len(df)} rows · parts {sorted(df.part.unique())}")
print(f"machines: {sorted(df.machine_id.dropna().unique())}")

## 2 · Shared style

Colours are a CVD-validated categorical set (worst adjacent pair ΔE 23.8 under deuteranopia);
each index also gets its own marker, so identity never rests on colour alone. One linear axis
for recall, one log axis for throughput — never a second y-scale.

In [ ]:
# Validated categorical palette: blue / green / purple / crimson.
PAL = {"hnsw": "#2563eb", "ivf_flat": "#059669", "ivf_pq": "#9333ea",
       "exact_flat": "#dc143c"}
MARK = {"hnsw": "o", "ivf_flat": "s", "ivf_pq": "^", "exact_flat": "D"}
LABEL = {"hnsw": "HNSW", "ivf_flat": "IVF-Flat", "ivf_pq": "IVF-PQ",
         "exact_flat": "exact (flat)"}
DS_LABEL = {"sift": "SIFT-1M (128-d, Euclidean)",
            "glove": "GloVe-100 (1.18M, angular, unit-normalised)"}

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 200, "font.size": 9,
    "axes.grid": True, "grid.alpha": 0.22, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 10, "legend.frameon": False, "legend.fontsize": 7.5,
})
FIG = f"{C.OUT}/figures"
TAB = f"{C.OUT}/results"

def at_recall(g, target):
    '''QPS at exactly `target` recall, log-interpolated along one swept curve.'''
    g = g.sort_values("recall_at_10")
    r, q = g.recall_at_10.values, g.qps_median.values
    ok = ~np.isnan(r) & ~np.isnan(q)
    r, q = r[ok], q[ok]
    if len(r) == 0 or r.max() < target:
        return None
    if r.min() >= target:
        return float(q.max())
    i = int(np.searchsorted(r, target))
    if i == 0 or r[i] == r[i - 1]:
        return float(q[i])
    w = (target - r[i - 1]) / (r[i] - r[i - 1])
    return float(math.exp(math.log(q[i - 1]) * (1 - w) + math.log(q[i]) * w))

def envelope(g):
    '''Upper envelope over build settings: the best QPS available at >= each recall.'''
    g = g.sort_values("recall_at_10", ascending=False)
    best, out = -np.inf, []
    for _, row in g.iterrows():
        if row.qps_median > best:
            best = row.qps_median
            out.append((row.recall_at_10, row.qps_median))
    return sorted(out)

print("style ready")

## 3 · Figures 1–2 — the Part 2 trade-off curves

Faint lines are individual build settings; the bold line is each index's **upper envelope**,
i.e. the best throughput reachable at or above a given recall. The envelope is what the
assignment means by "the best trade-off curve for each index".

In [ ]:
p2 = df[df.part == "2"]
figs = {}

for ds in ("sift", "glove"):
    d = p2[p2.dataset == ds]
    if not len(d):
        print(f"no Part 2 rows for {ds}"); continue
    fig, ax = plt.subplots(figsize=(6.9, 3.95))

    for kind, grp in d.groupby("index_kind"):
        col = PAL.get(kind, "gray")
        for bp, sub in grp.groupby("build_params"):
            sub = sub.sort_values("recall_at_10")
            ax.plot(sub.recall_at_10, sub.qps_median, "-", color=col, alpha=0.28, lw=0.9,
                    zorder=1)
        env = envelope(grp)
        if env:
            ax.plot([p[0] for p in env], [p[1] for p in env], "-", marker=MARK.get(kind, "x"),
                    color=col, lw=2.0, ms=5.5, mec="white", mew=0.7,
                    label=LABEL.get(kind, kind), zorder=3)

    ax.set_yscale("log")
    # The legend goes top-right: the curves run top-left to bottom-right, and the exact
    # baseline occupies the bottom edge, so a lower-left legend gets struck through by it.
    ax.legend(loc="upper right")

    ex = df[(df.part == "1") & (df.dataset == ds)
            & (df.search_param_value.astype(str) == "one_at_a_time")]
    if len(ex):
        eq = float(ex.qps_median.iloc[0])
        ax.axhline(eq, color=PAL["exact_flat"], ls="--", lw=1.4, zorder=2)
        ax.plot([1.0], [eq], MARK["exact_flat"], color=PAL["exact_flat"], ms=7,
                mec="white", mew=0.8, zorder=4)
        # Direct-labelled rather than put in the legend, so the line cannot cross its own text.
        x0 = ax.get_xlim()[0]
        ax.annotate(f"exact search — {eq:.1f} qps at recall 1.0",
                    xy=(x0, eq), xytext=(4, 5), textcoords="offset points",
                    ha="left", va="bottom", fontsize=7.5, color=PAL["exact_flat"])

    ax.axvline(0.95, color="#52525b", ls=":", lw=1.0, zorder=0)
    # Low on the axis: the top is taken by the legend, the bottom band is empty.
    ax.annotate("recall 0.95", xy=(0.95, 0.06), xycoords=("data", "axes fraction"),
                xytext=(-4, 0), textcoords="offset points", ha="right", va="bottom",
                fontsize=7, color="#52525b", rotation=90)
    ax.set_xlabel("recall@10  (tie-aware, exact distances recomputed)")
    ax.set_ylabel("queries / second   (1 thread, one query at a time)")
    ax.set_title(f"{DS_LABEL[ds]}\nrecall vs throughput — faint: build settings, bold: best envelope")
    ax.yaxis.set_minor_formatter(NullFormatter())
    fig.tight_layout()
    name = f"fig{'1' if ds == 'sift' else '2'}_{ds}_tradeoff.png"
    fig.savefig(f"{FIG}/{name}", bbox_inches="tight")
    figs[ds] = name
    plt.show()

## 4 · Table 1 — QPS at recall 0.90 and 0.95

In [ ]:
rows = []
for ds in ("sift", "glove"):
    d = p2[p2.dataset == ds]
    for kind in ("hnsw", "ivf_flat", "ivf_pq"):
        grp = d[d.index_kind == kind]
        if not len(grp):
            continue
        best = {}
        for t in (0.90, 0.95):
            cand = [(at_recall(sub, t), bp) for bp, sub in grp.groupby("build_params")]
            cand = [(q, bp) for q, bp in cand if q is not None]
            best[t] = max(cand) if cand else (None, None)
        rows.append(dict(
            dataset=ds, index=LABEL[kind],
            qps_at_090=best[0.90][0], setting_090=best[0.90][1],
            qps_at_095=best[0.95][0], setting_095=best[0.95][1],
            peak_recall=float(grp.recall_at_10.max())))
    ex = df[(df.part == "1") & (df.dataset == ds)
            & (df.search_param_value.astype(str) == "one_at_a_time")]
    if len(ex):
        rows.append(dict(dataset=ds, index="exact (flat)",
                         qps_at_090=float(ex.qps_median.iloc[0]),
                         setting_090="full scan",
                         qps_at_095=float(ex.qps_median.iloc[0]),
                         setting_095="full scan", peak_recall=1.0))

t1 = pd.DataFrame(rows)
t1.to_csv(f"{TAB}/table1_qps_at_recall.csv", index=False)

lines = ["| dataset | index | qps @ 0.90 | qps @ 0.95 | best setting @ 0.95 | peak recall |",
         "|---|---|---:|---:|---|---:|"]
for _, r in t1.iterrows():
    def fm(v):
        return f"{v:,.0f}" if pd.notna(v) else "never reached"
    st = r.setting_095 if isinstance(r.setting_095, str) else "—"
    lines.append(f"| {r.dataset} | {r['index']} | {fm(r.qps_at_090)} | {fm(r.qps_at_095)} "
                 f"| `{st}` | {r.peak_recall:.4f} |")
t1_md = "\n".join(lines)
open(f"{TAB}/table1_qps_at_recall.md", "w").write(t1_md)
print(t1_md)

## 5 · Table 2 — build time, index size and bytes per vector at recall 0.95

In [ ]:
rows = []
for ds in ("sift", "glove"):
    d = p2[p2.dataset == ds]
    n = int(d.n_vectors.iloc[0]) if len(d) else None
    dim = int(d.dim.iloc[0]) if len(d) else None
    for kind in ("hnsw", "ivf_flat", "ivf_pq"):
        grp = d[d.index_kind == kind]
        if not len(grp):
            continue
        cand = [(at_recall(sub, 0.95), bp) for bp, sub in grp.groupby("build_params")]
        cand = [(q, bp) for q, bp in cand if q is not None]
        if cand:
            _, bp = max(cand)
            sub = grp[grp.build_params == bp]
            note = ""
        else:   # never reached 0.95: report the setting that got closest
            bp = grp.loc[grp.recall_at_10.idxmax(), "build_params"]
            sub = grp[grp.build_params == bp]
            note = f"never reached 0.95 (peak {grp.recall_at_10.max():.3f})"
        sz = float(sub.index_size_bytes.dropna().iloc[0])
        rows.append(dict(dataset=ds, index=LABEL[kind], setting=bp,
                         build_s=float(sub.build_time_s.dropna().iloc[0]),
                         size_mb=sz / 1e6, bytes_per_vector=sz / n, note=note))
    if n:
        rows.append(dict(dataset=ds, index="exact (flat)", setting="—", build_s=0.4,
                         size_mb=n * dim * 4 / 1e6, bytes_per_vector=dim * 4, note=""))

t2 = pd.DataFrame(rows)
t2.to_csv(f"{TAB}/table2_build_memory.csv", index=False)
lines = ["| dataset | index | setting | build (s) | size (MB) | bytes/vector | note |",
         "|---|---|---|---:|---:|---:|---|"]
for _, r in t2.iterrows():
    bs = f"{r.build_s:,.0f}" if r.build_s >= 10 else f"{r.build_s:.1f}"
    lines.append(f"| {r.dataset} | {r['index']} | `{r.setting}` | {bs} "
                 f"| {r.size_mb:,.0f} | {r.bytes_per_vector:,.0f} | {r.note} |")
t2_md = "\n".join(lines)
open(f"{TAB}/table2_build_memory.md", "w").write(t2_md)
print(t2_md)

## 6 · Figure 3 — Part 3 scaling

In [ ]:
p3 = df[df.part == "3"]
SIZES = sorted(p3.n_vectors.unique()) if len(p3) else []
best3 = {}
for kind in ("exact_flat", "ivf_flat", "hnsw"):
    for n in SIZES:
        g = p3[(p3.index_kind == kind) & (p3.n_vectors == n)]
        if len(g):
            best3[(kind, n)] = g.loc[g.qps_median.idxmax()]

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.3))
EXP = {}
ax = axes[0]
for kind in ("exact_flat", "ivf_flat", "hnsw"):
    pts = [(n, best3[(kind, n)].qps_median) for n in SIZES if (kind, n) in best3]
    if len(pts) < 2:
        continue
    xs, ys = zip(*pts)
    a = float(np.polyfit(np.log(xs), np.log(ys), 1)[0])
    EXP[kind] = a
    ax.plot(xs, ys, "-", marker=MARK[kind], color=PAL[kind], lw=2, ms=6,
            mec="white", mew=0.7, label=f"{LABEL[kind]}   $n^{{{a:+.2f}}}$")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xticks(SIZES); ax.set_xticklabels([f"{int(n/1000)}K" if n < 1e6 else "1M" for n in SIZES])
ax.set_xlabel("vectors indexed"); ax.set_ylabel("queries / second at recall@10 = 0.95")
ax.set_title("(a) cost of recall 0.95 vs dataset size")
ax.legend(loc="lower left")
ax.xaxis.set_minor_formatter(NullFormatter())

ax = axes[1]
for kind in ("ivf_flat", "hnsw"):
    pts = [(n, best3[(kind, n)].build_time_s) for n in SIZES
           if (kind, n) in best3 and pd.notna(best3[(kind, n)].build_time_s)]
    if len(pts) < 2:
        continue
    xs, ys = zip(*pts)
    ax.plot(xs, ys, "-", marker=MARK[kind], color=PAL[kind], lw=2, ms=6,
            mec="white", mew=0.7, label=LABEL[kind])
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xticks(SIZES); ax.set_xticklabels([f"{int(n/1000)}K" if n < 1e6 else "1M" for n in SIZES])
ax.set_xlabel("vectors indexed"); ax.set_ylabel("build time (s, 4 threads)")
ax.set_title("(b) build cost vs dataset size")
ax.legend(loc="upper left")
ax.xaxis.set_minor_formatter(NullFormatter())

fig.suptitle("Part 3 — SIFT scaling: exact search pays linearly, HNSW barely notices",
             fontsize=10.5, y=1.02)
fig.tight_layout()
fig.savefig(f"{FIG}/fig3_part3_scaling.png", bbox_inches="tight")
plt.show()
print({k: round(v, 3) for k, v in EXP.items()})

## 7 · Figure 4 — Part 4 geometry

Re-rendered here from `results.csv` so it shares the report's style and palette. `uniform` and
`clustered` are the two conditions; marker shape distinguishes the index.

In [ ]:
p4 = df[df.part == "4"].copy()

# scanned_frac and rc_at_10 only became real columns partway through Part 4, so older rows
# carry them inside `notes`. Backfill per-row from notes, then -- crucially -- fill
# rc_at_10 ACROSS each dataset's rows: relative contrast is a property of the dataset, and
# the sweep rows never recorded it. Reading it off whichever row happens to come first
# silently loses d=1024, whose informative row is the one dedup drops.
for col, pat in (("scanned_frac", r"scanned_frac=([0-9.]+)"), ("rc_at_10", r"rc10=([0-9.]+)")):
    have = pd.to_numeric(p4[col], errors="coerce") if col in p4.columns else np.nan
    from_notes = pd.to_numeric(
        p4.notes.astype(str).str.extract(pat)[0], errors="coerce")
    p4[col] = (have.fillna(from_notes) if isinstance(have, pd.Series) else from_notes)

# Relative contrast is a property of the DATASET, and only the fixed-budget rows recorded
# it -- which at d=1024 are exactly the rows dedup discards (they were logged twice, and
# the surviving copy carries sweep-style notes). So harvest it from the raw table, before
# deduplication, and map it on by dataset name.
_r4 = raw[raw.part.astype(str) == "4"]
_rc_map = (pd.to_numeric(_r4.get("rc_at_10"), errors="coerce")
           .fillna(pd.to_numeric(_r4.notes.astype(str).str.extract(r"rc10=([0-9.]+)")[0],
                                 errors="coerce"))
           .groupby(_r4.dataset).first())
p4["rc_at_10"] = p4["rc_at_10"].fillna(p4.dataset.map(_rc_map))
assert p4.rc_at_10.notna().all(), "a Part 4 dataset has no relative-contrast value anywhere"
assert p4.scanned_frac.notna().all(), "a Part 4 row has no scanned fraction"
p4["cond"] = np.where(p4.dataset.str.contains("clustered"), "clustered", "uniform")
P4COL = {"uniform": "#dc2626", "clustered": "#2563eb"}
DIMS4 = sorted(p4.dim.unique())
FIXEDV = {"hnsw": 64, "ivf_flat": 16}

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))

ax = axes[0]
for cond in ("uniform", "clustered"):
    pts = []
    for d in DIMS4:
        g = p4[(p4.cond == cond) & (p4.dim == d)]
        if len(g):
            pts.append((d, float(g.rc_at_10.dropna().iloc[0])))
    if pts:
        xs, ys = zip(*pts)
        ax.plot(xs, ys, "-o", color=P4COL[cond], lw=2, ms=6, mec="white", mew=0.7, label=cond)
for y, nm in ((2.63, "SIFT"), (1.51, "GloVe")):
    ax.axhline(y, color="#71717a", ls=":", lw=0.9)
    ax.text(DIMS4[0] * 1.08, y * 1.03, nm, fontsize=7, color="#71717a")
ax.axhline(1.0, color="#27272a", lw=0.9)
ax.set_xscale("log", base=2); ax.set_xticks(DIMS4); ax.set_xticklabels(DIMS4)
ax.set_xlabel("dimension"); ax.set_ylabel("relative contrast RC@10")
ax.set_title("(a) the manipulation\ncontrast collapses — uniform only")
ax.legend(loc="center right"); ax.xaxis.set_minor_formatter(NullFormatter())

ax = axes[1]
for cond in ("uniform", "clustered"):
    for ik in ("hnsw", "ivf_flat"):
        pts = []
        for d in DIMS4:
            g = p4[(p4.cond == cond) & (p4.dim == d) & (p4.index_kind == ik)
                   & (p4.search_param_value.astype(float) == FIXEDV[ik])]
            if len(g):
                pts.append((d, float(g.recall_at_10.iloc[0])))
        if pts:
            xs, ys = zip(*pts)
            ax.plot(xs, ys, "-" if ik == "hnsw" else "--", marker=MARK[ik],
                    color=P4COL[cond], lw=1.9, ms=6, mec="white", mew=0.7,
                    label=f"{LABEL[ik]} · {cond}")
ax.axhline(0.95, color="#52525b", ls=":", lw=1.0)
ax.set_xscale("log", base=2); ax.set_xticks(DIMS4); ax.set_xticklabels(DIMS4)
ax.set_ylim(-0.04, 1.06)
ax.set_xlabel("dimension"); ax.set_ylabel("recall@10 at a fixed budget")
ax.set_title("(b) same budget everywhere (ef=64, nprobe=16)\nrecall tracks contrast, not dimension")
ax.legend(loc="lower left"); ax.xaxis.set_minor_formatter(NullFormatter())

ax = axes[2]
dmax = DIMS4[-1]
for cond in ("uniform", "clustered"):
    for ik in ("hnsw", "ivf_flat"):
        g = p4[(p4.cond == cond) & (p4.dim == dmax) & (p4.index_kind == ik)]
        g = g.dropna(subset=["scanned_frac"]).sort_values("scanned_frac")
        if len(g):
            ax.plot(g.scanned_frac * 100, g.recall_at_10, "-" if ik == "hnsw" else "--",
                    marker=MARK[ik], color=P4COL[cond], lw=1.9, ms=5.5, mec="white", mew=0.7,
                    label=f"{LABEL[ik]} · {cond}")
ax.axhline(0.95, color="#52525b", ls=":", lw=1.0)
ax.axvline(50, color="#dc143c", ls="--", lw=1.2)
ax.text(52, 0.06, "50% of data", fontsize=7, color="#dc143c", rotation=90)
ax.set_xscale("log"); ax.set_ylim(-0.04, 1.06)
ax.set_xlabel("% of the dataset scanned per query")
ax.set_ylabel("recall@10")
ax.set_title(f"(c) at d={dmax}: recall 0.95 costs\na near-exhaustive scan")
ax.legend(loc="upper left"); ax.xaxis.set_minor_formatter(NullFormatter())

fig.suptitle("Part 4 — distance concentration breaks both indexes; the clustered control at "
             "the same dimension does not", fontsize=10.5, y=1.03)
fig.tight_layout()
fig.savefig(f"{FIG}/fig4_part4_geometry.png", bbox_inches="tight")
plt.show()

## 8 · Numbers the report's prose quotes

In [ ]:
# The machine block must describe the machine the MEASUREMENTS were taken on, not whatever
# machine happens to be running this aggregation. Reading it from env_fingerprint() here is
# wrong whenever the figures are re-rendered elsewhere, and silently attributes someone
# else's hardware to the results.
_mid = str(df.machine_id.dropna().mode().iloc[0])
_env_path = next((p for p in (f"{C.OUT}/env/{_mid}.json", f"{C.OUT}/logs/env/{_mid}.json")
                  if os.path.exists(p)), None)
if _env_path is None:
    try:
        _env_path = C.fetch(f"logs/env/{_mid}.json", TOKEN, verbose=False)
    except Exception:
        _env_path = None
if _env_path:
    _mach = json.load(open(_env_path))
    print(f"machine block: from the recorded fingerprint of {_mid}")
else:
    _mach = {k: ENV.get(k) for k in ("cpu_model", "cpu_logical", "mem_gb", "gpu",
                                     "numpy", "python", "machine_id")}
    print(f"!! no recorded fingerprint for {_mid}; falling back to THIS machine, which is "
          f"only correct if they are the same")
# These two always come from the rows themselves.
_mach["cpu_model"] = str(df.cpu_model.dropna().mode().iloc[0])
_mach["faiss"] = str(df.faiss_version.dropna().mode().iloc[0])
_mach["machine_id"] = _mid
if ENV.get("machine_id") != _mid:
    print(f"   (this aggregation is running on {ENV.get('machine_id')}, not {_mid} — "
          f"the figures are rendered here but the measurements are not)")
NUM = {"machine": _mach,
       "rows_total": int(len(df)), "duplicates_dropped": int(len(raw) - len(df)),
       "scaling_exponents": {k: round(v, 3) for k, v in EXP.items()}}

for ds in ("sift", "glove"):
    d = p2[p2.dataset == ds]
    ex = df[(df.part == "1") & (df.dataset == ds)]
    e1 = ex[ex.search_param_value.astype(str) == "one_at_a_time"]
    eb = ex[ex.search_param_value.astype(str) == "batch"]
    blk = {"exact_qps_1q": float(e1.qps_median.iloc[0]) if len(e1) else None,
           "exact_qps_batch": float(eb.qps_median.iloc[0]) if len(eb) else None}
    if blk["exact_qps_1q"] and blk["exact_qps_batch"]:
        blk["batch_speedup"] = round(blk["exact_qps_batch"] / blk["exact_qps_1q"], 1)
    for kind in ("hnsw", "ivf_flat", "ivf_pq"):
        grp = d[d.index_kind == kind]
        if not len(grp):
            continue
        cand = [(at_recall(s, 0.95), bp) for bp, s in grp.groupby("build_params")]
        cand = [(q, bp) for q, bp in cand if q is not None]
        blk[kind] = {"qps_at_095": round(max(cand)[0], 1) if cand else None,
                     "best_setting": max(cand)[1] if cand else None,
                     "peak_recall": round(float(grp.recall_at_10.max()), 4)}
    if blk.get("hnsw", {}).get("qps_at_095") and blk.get("ivf_flat", {}).get("qps_at_095"):
        blk["hnsw_over_ivf"] = round(blk["hnsw"]["qps_at_095"] / blk["ivf_flat"]["qps_at_095"], 2)
    if blk.get("hnsw", {}).get("qps_at_095") and blk["exact_qps_1q"]:
        blk["hnsw_over_exact"] = round(blk["hnsw"]["qps_at_095"] / blk["exact_qps_1q"], 0)
    NUM[ds] = blk

p4f = {}
for cond in ("uniform", "clustered"):
    for ik in ("hnsw", "ivf_flat"):
        seq = []
        for d_ in DIMS4:
            g = p4[(p4.cond == cond) & (p4.dim == d_) & (p4.index_kind == ik)
                   & (p4.search_param_value.astype(float) == FIXEDV[ik])]
            seq.append(round(float(g.recall_at_10.iloc[0]), 4) if len(g) else None)
        p4f[f"{cond}_{ik}"] = seq
NUM["part4_fixed_budget_recall"] = {"dims": [int(x) for x in DIMS4], **p4f}
NUM["part4_rc10"] = {
    cond: [round(float(p4[(p4.cond == cond) & (p4.dim == d_)].rc_at_10.dropna().iloc[0]), 3)
           for d_ in DIMS4] for cond in ("uniform", "clustered")}

with open(f"{TAB}/report_numbers.json", "w") as f:
    json.dump(NUM, f, indent=2)
print(json.dumps(NUM, indent=2)[:3000])

## 9 · Upload and summary

In [ ]:
arte = ([f"{FIG}/{n}" for n in figs.values()]
        + [f"{FIG}/fig3_part3_scaling.png", f"{FIG}/fig4_part4_geometry.png"]
        + [f"{TAB}/table1_qps_at_recall.md", f"{TAB}/table1_qps_at_recall.csv",
           f"{TAB}/table2_build_memory.md", f"{TAB}/table2_build_memory.csv",
           f"{TAB}/report_numbers.json"])
for a in arte:
    if os.path.exists(a):
        sub = "figures" if a.endswith(".png") else "results"
        C.hf_upload(a, f"{sub}/{os.path.basename(a)}", token=TOKEN, msg="nb06 report assets")

print("\n" + "=" * 80); print("nb06 SUMMARY (report assets)"); print("=" * 80)
print(f"rows used {len(df)} (dropped {len(raw) - len(df)} duplicates) · "
      f"machines {sorted(df.machine_id.dropna().unique())}")
print(f"\nfiles written to {C.OUT}"
      + (" and mirrored to HF:" if C.hf_enabled() else " (HF mirroring off):"))
for a in arte:
    if os.path.exists(a):
        print(f"  {os.path.basename(a):32s} {C.human(os.path.getsize(a))}")
print(f"\nscaling exponents: " + ", ".join(f"{k} {v:+.3f}" for k, v in EXP.items()))
print("\nheadline numbers")
for ds in ("sift", "glove"):
    b = NUM.get(ds, {})
    if b.get("hnsw"):
        print(f"  {ds:6s} exact {b['exact_qps_1q']:.1f} qps · "
              f"HNSW@0.95 {b['hnsw']['qps_at_095']:.0f} · "
              f"IVF-Flat@0.95 {b['ivf_flat']['qps_at_095']:.0f} · "
              f"PQ peak {b['ivf_pq']['peak_recall']:.3f} · "
              f"HNSW/IVF {b.get('hnsw_over_ivf')}x · HNSW/exact {b.get('hnsw_over_exact'):.0f}x")
print("=" * 80)

### Done measuring

Everything the report needs now exists as a file. The remaining work is local: assembling
`code/` into the required layout (`run_all.sh` with a `--quick` path, `README.md`,
`requirements.txt`, `tests/`), writing `report.pdf`, filling `ai_log/`, and zipping.

Download `/kaggle/working` (or pull from the HF repo) and paste this summary back.